In [1]:
import os
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================

# Change this to wherever your KT1 folder is located
KT1_PATH = Path("D:\Lokesh files\Projects\LearningDNA\KT1")

# ============================================================
# CHECK DATASET
# ============================================================

print("KT1 path:", KT1_PATH)
print("Exists:", KT1_PATH.exists())

if not KT1_PATH.exists():
    raise FileNotFoundError(
        f"KT1 folder not found: {KT1_PATH}\n"
        "Please update KT1_PATH."
    )

# Find CSV files
csv_files = list(KT1_PATH.glob("*.csv"))

print("Number of CSV files found:", len(csv_files))

# Display first few files
print("\nFirst 10 files:")
for file in csv_files[:10]:
    print(file.name)

<>:10: SyntaxWarning: invalid escape sequence '\L'
<>:10: SyntaxWarning: invalid escape sequence '\L'
C:\Users\Lokesh S\AppData\Local\Temp\ipykernel_6232\1000034014.py:10: SyntaxWarning: invalid escape sequence '\L'
  KT1_PATH = Path("D:\Lokesh files\Projects\LearningDNA\KT1")


KT1 path: D:\Lokesh files\Projects\LearningDNA\KT1
Exists: True
Number of CSV files found: 784309

First 10 files:
u1.csv
u10.csv
u100.csv
u1000.csv
u10000.csv
u100000.csv
u100001.csv
u100002.csv
u100003.csv
u100004.csv


In [2]:
# Pick the first student file
sample_file = csv_files[0]

print("Reading:", sample_file.name)

student_df = pd.read_csv(sample_file)

print("\nShape:")
print(student_df.shape)

print("\nColumns:")
print(student_df.columns.tolist())

print("\nFirst 10 interactions:")
display(student_df.head(10))

print("\nData types:")
print(student_df.dtypes)

Reading: u1.csv

Shape:
(1082, 5)

Columns:
['timestamp', 'solving_id', 'question_id', 'user_answer', 'elapsed_time']

First 10 interactions:


,timestamp,solving_id,question_id,user_answer,elapsed_time
0,1565096190868,1,q5012,b,38000
1,1565096221062,2,q4706,c,24000
2,1565096293432,3,q4366,b,68000
3,1565096339668,4,q4829,a,42000
4,1565096401774,5,q6528,b,59000
5,1565096463370,6,q4793,a,58000
6,1565096501746,7,q6488,a,35000
7,1565097101361,8,q356,b,23000
8,1565097171393,9,q1382,c,22000
9,1565097240758,10,q830,b,25000



Data types:
timestamp        int64
solving_id       int64
question_id     object
user_answer     object
elapsed_time     int64
dtype: object


In [3]:
student_id = sample_file.stem

student_df["user_id"] = student_id

# Put user_id first
student_df = student_df[
    ["user_id"] +
    [col for col in student_df.columns if col != "user_id"]
]

display(student_df.head(10))

,user_id,timestamp,solving_id,question_id,user_answer,elapsed_time
0,u1,1565096190868,1,q5012,b,38000
1,u1,1565096221062,2,q4706,c,24000
2,u1,1565096293432,3,q4366,b,68000
3,u1,1565096339668,4,q4829,a,42000
4,u1,1565096401774,5,q6528,b,59000
5,u1,1565096463370,6,q4793,a,58000
6,u1,1565096501746,7,q6488,a,35000
7,u1,1565097101361,8,q356,b,23000
8,u1,1565097171393,9,q1382,c,22000
9,u1,1565097240758,10,q830,b,25000


In [4]:
print("========== DATASET SUMMARY ==========")

print("Student ID:", student_id)
print("Number of interactions:", len(student_df))

print("\n---------- Missing Values ----------")
print(student_df.isnull().sum())

print("\n---------- Unique Values ----------")
print("Questions:", student_df["question_id"].nunique())
print("Sessions/Bundles:", student_df["solving_id"].nunique())
print("Answers:", student_df["user_answer"].nunique())

print("\n---------- Answer Distribution ----------")
print(student_df["user_answer"].value_counts())

print("\n---------- Elapsed Time ----------")
print(student_df["elapsed_time"].describe())

print("\n---------- Timestamp ----------")
print(student_df["timestamp"].describe())

========== DATASET SUMMARY ==========
Student ID: u1
Number of interactions: 1082

---------- Missing Values ----------
user_id         0
timestamp       0
solving_id      0
question_id     0
user_answer     0
elapsed_time    0
dtype: int64

---------- Unique Values ----------
Questions: 926
Sessions/Bundles: 671
Answers: 4

---------- Answer Distribution ----------
user_answer
b    311
a    289
c    276
d    206
Name: count, dtype: int64

---------- Elapsed Time ----------
count    1.082000e+03
mean     4.957844e+04
std      5.839036e+04
min      6.660000e+02
25%      2.425000e+04
50%      3.675000e+04
75%      5.325000e+04
max      1.200000e+06
Name: elapsed_time, dtype: float64

---------- Timestamp ----------
count    1.082000e+03
mean     1.568308e+12
std      1.372201e+09
min      1.565096e+12
25%      1.567346e+12
50%      1.568809e+12
75%      1.569488e+12
max      1.569647e+12
Name: timestamp, dtype: float64


In [5]:
# Sort chronologically
student_df = student_df.sort_values("timestamp").reset_index(drop=True)

# Timestamp in datetime format
student_df["datetime"] = pd.to_datetime(
    student_df["timestamp"],
    unit="ms"
)

# Elapsed time is in milliseconds
student_df["elapsed_seconds"] = (
    student_df["elapsed_time"] / 1000.0
)

# Time between consecutive interactions
student_df["time_delta_seconds"] = (
    student_df["timestamp"].diff() / 1000.0
)

# First interaction has no previous interaction
student_df["time_delta_seconds"] = (
    student_df["time_delta_seconds"].fillna(0)
)

display(
    student_df[
        [
            "user_id",
            "timestamp",
            "datetime",
            "question_id",
            "user_answer",
            "elapsed_seconds",
            "time_delta_seconds"
        ]
    ].head(15)
)

,user_id,timestamp,datetime,question_id,user_answer,elapsed_seconds,time_delta_seconds
0,u1,1565096190868,2019-08-06 12:56:30.868,q5012,b,38.0,0.000
1,u1,1565096221062,2019-08-06 12:57:01.062,q4706,c,24.0,30.194
2,u1,1565096293432,2019-08-06 12:58:13.432,q4366,b,68.0,72.370
3,u1,1565096339668,2019-08-06 12:58:59.668,q4829,a,42.0,46.236
4,u1,1565096401774,2019-08-06 13:00:01.774,q6528,b,59.0,62.106
5,u1,1565096463370,2019-08-06 13:01:03.370,q4793,a,58.0,61.596
6,u1,1565096501746,2019-08-06 13:01:41.746,q6488,a,35.0,38.376
7,u1,1565097101361,2019-08-06 13:11:41.361,q356,b,23.0,599.615
8,u1,1565097171393,2019-08-06 13:12:51.393,q1382,c,22.0,70.032
9,u1,1565097240758,2019-08-06 13:14:00.758,q830,b,25.0,69.365


In [6]:
import numpy as np

student_df["log_elapsed_time"] = np.log1p(
    student_df["elapsed_seconds"]
)

student_df["log_time_delta"] = np.log1p(
    student_df["time_delta_seconds"]
)

display(
    student_df[
        [
            "elapsed_seconds",
            "log_elapsed_time",
            "time_delta_seconds",
            "log_time_delta"
        ]
    ].head(15)
)

,elapsed_seconds,log_elapsed_time,time_delta_seconds,log_time_delta
0,38.0,3.663562,0.000,0.000000
1,24.0,3.218876,30.194,3.440226
2,68.0,4.234107,72.370,4.295515
3,42.0,3.761200,46.236,3.855156
4,59.0,4.094345,62.106,4.144816
5,58.0,4.077537,61.596,4.136701
6,35.0,3.583519,38.376,3.673156
7,23.0,3.178054,599.615,6.397954
8,22.0,3.135494,70.032,4.263130
9,25.0,3.258097,69.365,4.253696


In [7]:
print("========== DATA QUALITY CHECK ==========")

print("Negative elapsed times:",
      (student_df["elapsed_time"] < 0).sum())

print("Zero elapsed times:",
      (student_df["elapsed_time"] == 0).sum())

print("Negative time deltas:",
      (student_df["time_delta_seconds"] < 0).sum())

print("Missing question IDs:",
      student_df["question_id"].isna().sum())

print("Missing answers:",
      student_df["user_answer"].isna().sum())

print("Duplicate rows:",
      student_df.duplicated().sum())

========== DATA QUALITY CHECK ==========
Negative elapsed times: 0
Zero elapsed times: 0
Negative time deltas: 0
Missing question IDs: 0
Missing answers: 0
Duplicate rows: 0


In [8]:
import os
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================

# Change this to wherever your KT1 folder is located
CONTENT_PATH = Path("D:\Lokesh files\Projects\LearningDNA\contents")

# ============================================================
# CHECK DATASET
# ============================================================

print("Contents path:", CONTENT_PATH)
print("Exists:", CONTENT_PATH.exists())

if not CONTENT_PATH.exists():
    raise FileNotFoundError(
        f"KT1 folder not found: {CONTENT_PATH}\n"
        "Please update CONTENT_PATH."
    )

# Find CSV files
csv_files = list(CONTENT_PATH.glob("*.csv"))

print("Number of CSV files found:", len(csv_files))

for file in csv_files:
    print(file.name)

Contents path: D:\Lokesh files\Projects\LearningDNA\contents
Exists: True
Number of CSV files found:

<>:10: SyntaxWarning: invalid escape sequence '\L'
<>:10: SyntaxWarning: invalid escape sequence '\L'
C:\Users\Lokesh S\AppData\Local\Temp\ipykernel_6232\3837369255.py:10: SyntaxWarning: invalid escape sequence '\L'
  CONTENT_PATH = Path("D:\Lokesh files\Projects\LearningDNA\contents")


 4
coupons.csv
lectures.csv
payments.csv
questions.csv


In [9]:
# Observing all the files in the content folder
for i in range(len(csv_files)):
    sample_file = csv_files[i]
    
    print("Reading:", sample_file.name)
    
    student_df = pd.read_csv(sample_file)
    
    print("\nShape:")
    print(student_df.shape)
    
    print("\nColumns:")
    print(student_df.columns.tolist())
    
    print("\nFirst 10 interactions:")
    display(student_df.head(10))
    
    print("\nData types:")
    print(student_df.dtypes)

    print("*"*40)
    print('\n')

Reading: coupons.csv

Shape:
(91, 3)

Columns:
['coupon_id', 'coupon_type', 'duration']

First 10 interactions:


,coupon_id,coupon_type,duration
0,c1,type-0,4320000000
1,c2,type-0,5184000000
2,c3,type-0,7776000000
3,c4,type-0,8640000000
4,c5,type-0,15552000000
5,c6,type-0,17280000000
6,c7,type-1,86400000
7,c8,type-1,172800000
8,c9,type-1,1296000000
9,c10,type-1,1382400000



Data types:
coupon_id      object
coupon_type    object
duration        int64
dtype: object
****************************************


Reading: lectures.csv

Shape:
(1021, 5)

Columns:
['lecture_id', 'part', 'tags', 'video_length', 'deployed_at']

First 10 interactions:


,lecture_id,part,tags,video_length,deployed_at
0,l520,5,142,-1,-1
1,l592,6,142,-1,-1
2,l1259,1,222,359000,1570424729123
3,l1260,1,220,487000,1570424738105
4,l1261,1,221,441000,1570424743162
5,l1262,1,223,587000,1570424748780
6,l1263,1,224,490000,1570424755399
7,l1264,2,227,298000,1570424755678
8,l1265,2,231,303000,1570424760727
9,l1266,2,229,345000,1570424765161



Data types:
lecture_id      object
part             int64
tags             int64
video_length     int64
deployed_at      int64
dtype: object
****************************************


Reading: payments.csv

Shape:
(190, 4)

Columns:
['payment_item_id', 'type', 'duaration', 'number_of_questions']

First 10 interactions:


,payment_item_id,type,duaration,number_of_questions
0,p1,pass,5184000000,-1
1,p2,pass,15552000000,-1
2,p3,pass,31536000000,-1
3,p4,paygo,-1,2000
4,p5,pass,5184000000,-1
5,p6,pass,15552000000,-1
6,p7,paygo,-1,4000
7,p8,pass,10368000000,-1
8,p9,pass,31536000000,-1
9,p10,pass,5184000000,-1



Data types:
payment_item_id        object
type                   object
duaration               int64
number_of_questions     int64
dtype: object
****************************************


Reading: questions.csv

Shape:
(13169, 7)

Columns:
['question_id', 'bundle_id', 'explanation_id', 'correct_answer', 'part', 'tags', 'deployed_at']

First 10 interactions:


,question_id,bundle_id,explanation_id,correct_answer,part,tags,deployed_at
0,q1,b1,e1,b,1,1;2;179;181,1558093217098
1,q2,b2,e2,a,1,15;2;182,1558093219720
2,q3,b3,e3,b,1,14;2;179;183,1558093222784
3,q4,b4,e4,b,1,9;2;179;184,1558093225357
4,q5,b5,e5,c,1,8;2;179;181,1558093228439
5,q6,b6,e6,d,1,9;2;179;182,1558093231307
6,q7,b7,e7,d,1,11;7;179;183,1558093233992
7,q8,b8,e8,b,1,20;21;179;184,1558093236876
8,q9,b9,e9,c,1,13;2;179;183,1558093239849
9,q10,b10,e10,c,1,17;7;182,1558093243018



Data types:
question_id       object
bundle_id         object
explanation_id    object
correct_answer    object
part               int64
tags              object
deployed_at        int64
dtype: object
****************************************




In [10]:
from pathlib import Path
import pandas as pd

# Change this to your contents folder
CONTENTS_PATH = Path(r"D:\Lokesh files\Projects\LearningDNA\contents")

questions_file = CONTENTS_PATH / "questions.csv"

print("Reading:", questions_file)

questions_df = pd.read_csv(questions_file)

print("\nShape:")
print(questions_df.shape)

print("\nColumns:")
print(questions_df.columns.tolist())

print("\nFirst 10 rows:")
display(questions_df.head(10))

print("\nData types:")
print(questions_df.dtypes)

print("\nMissing values:")
print(questions_df.isnull().sum())

Reading: D:\Lokesh files\Projects\LearningDNA\contents\questions.csv

Shape:
(13169, 7)

Columns:
['question_id', 'bundle_id', 'explanation_id', 'correct_answer', 'part', 'tags', 'deployed_at']

First 10 rows:


,question_id,bundle_id,explanation_id,correct_answer,part,tags,deployed_at
0,q1,b1,e1,b,1,1;2;179;181,1558093217098
1,q2,b2,e2,a,1,15;2;182,1558093219720
2,q3,b3,e3,b,1,14;2;179;183,1558093222784
3,q4,b4,e4,b,1,9;2;179;184,1558093225357
4,q5,b5,e5,c,1,8;2;179;181,1558093228439
5,q6,b6,e6,d,1,9;2;179;182,1558093231307
6,q7,b7,e7,d,1,11;7;179;183,1558093233992
7,q8,b8,e8,b,1,20;21;179;184,1558093236876
8,q9,b9,e9,c,1,13;2;179;183,1558093239849
9,q10,b10,e10,c,1,17;7;182,1558093243018



Data types:
question_id       object
bundle_id         object
explanation_id    object
correct_answer    object
part               int64
tags              object
deployed_at        int64
dtype: object

Missing values:
question_id       0
bundle_id         0
explanation_id    0
correct_answer    0
part              0
tags              0
deployed_at       0
dtype: int64


In [11]:
QUESTIONS_FILE = Path(r"D:\Lokesh files\Projects\LearningDNA\contents\questions.csv")
OUTPUT_DIR = Path("D:/Lokesh files/Projects/LearningDNA/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================

# Start small for testing
NUM_STUDENTS = 10

# ============================================================
# 1. LOAD QUESTION METADATA
# ============================================================

print("Loading question metadata...")

questions_df = pd.read_csv(QUESTIONS_FILE)

questions_df = questions_df[
    [
        "question_id",
        "correct_answer",
        "tags"
    ]
].copy()

print("Questions loaded:", len(questions_df))

# Check duplicate question IDs
duplicate_questions = questions_df["question_id"].duplicated().sum()

print("Duplicate question IDs:", duplicate_questions)

# ============================================================
# 2. FIND KT1 STUDENT FILES
# ============================================================

student_files = sorted(KT1_PATH.glob("u*.csv"))

print("KT1 student files found:", len(student_files))

if len(student_files) == 0:
    raise FileNotFoundError(
        "No KT1 student CSV files found. Check KT1_PATH."
    )

# Select only a small number for initial testing
student_files = student_files[:NUM_STUDENTS]

print("Processing:", len(student_files), "students")

# ============================================================
# 3. READ AND MERGE STUDENT DATA
# ============================================================

all_students = []

for file in student_files:

    user_id = file.stem

    df = pd.read_csv(file)

    # Add student ID
    df["user_id"] = user_id

    # --------------------------------------------------------
    # Join question metadata
    # --------------------------------------------------------

    df = df.merge(
        questions_df,
        on="question_id",
        how="left",
        validate="many_to_one"
    )

    # --------------------------------------------------------
    # Correctness
    # --------------------------------------------------------

    df["correctness"] = (
        df["user_answer"] == df["correct_answer"]
    ).astype("int8")

    # --------------------------------------------------------
    # Chronological ordering
    # --------------------------------------------------------

    df = df.sort_values(
        ["timestamp", "solving_id"]
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Attempt index
    #
    # Number of times this student has encountered
    # the same question up to this interaction.
    # --------------------------------------------------------

    df["attempt_index"] = (
        df.groupby("question_id")
        .cumcount()
        + 1
    )

    # --------------------------------------------------------
    # Time delta
    #
    # Time since previous interaction by this student.
    # --------------------------------------------------------

    df["time_delta_seconds"] = (
        df["timestamp"].diff() / 1000.0
    )

    # First interaction has no previous interaction
    df["time_delta_seconds"] = (
        df["time_delta_seconds"]
        .fillna(0)
    )

    # --------------------------------------------------------
    # Elapsed time
    #
    # Original elapsed_time is milliseconds.
    # Convert to seconds.
    # --------------------------------------------------------

    df["elapsed_seconds"] = (
        df["elapsed_time"] / 1000.0
    )

    # --------------------------------------------------------
    # Log-normalized temporal features
    # --------------------------------------------------------

    df["log_elapsed_time"] = np.log1p(
        df["elapsed_seconds"]
    )

    df["log_time_delta"] = np.log1p(
        df["time_delta_seconds"]
    )

    all_students.append(df)

# ============================================================
# 4. COMBINE STUDENTS
# ============================================================

processed_df = pd.concat(
    all_students,
    ignore_index=True
)

# ============================================================
# 5. SELECT FINAL COLUMNS
# ============================================================

processed_df = processed_df[
    [
        "user_id",
        "timestamp",
        "solving_id",
        "question_id",
        "user_answer",
        "correct_answer",
        "correctness",
        "tags",
        "elapsed_time",
        "elapsed_seconds",
        "attempt_index",
        "time_delta_seconds",
        "log_elapsed_time",
        "log_time_delta"
    ]
]

# ============================================================
# 6. VALIDATION
# ============================================================

print("\n========== PROCESSED DATA ==========")

print("Shape:", processed_df.shape)

print("\nColumns:")
print(processed_df.columns.tolist())

print("\nMissing values:")
print(processed_df.isnull().sum())

print("\nCorrectness distribution:")
print(processed_df["correctness"].value_counts())

print("\nUnique students:")
print(processed_df["user_id"].nunique())

print("\nUnique questions:")
print(processed_df["question_id"].nunique())

print("\nUnique tags:")
print(
    processed_df["tags"]
    .dropna()
    .str.split(";")
    .explode()
    .nunique()
)

print("\nFirst 10 processed rows:")
display(processed_df.head(10))

# ============================================================
# 7. SAVE
# ============================================================

output_file = OUTPUT_DIR / "kt1_processed_sample.csv"

processed_df.to_csv(
    output_file,
    index=False
)

print("\nSaved to:")
print(output_file.resolve())

Loading question metadata...
Questions loaded: 13169
Duplicate question IDs: 0
KT1 student files found: 784309
Processing: 10 students

========== PROCESSED DATA ==========
Shape: (3368, 14)

Columns:
['user_id', 'timestamp', 'solving_id', 'question_id', 'user_answer', 'correct_answer', 'correctness', 'tags', 'elapsed_time', 'elapsed_seconds', 'attempt_index', 'time_delta_seconds', 'log_elapsed_time', 'log_time_delta']

Missing values:
user_id               0
timestamp             0
solving_id            0
question_id           0
user_answer           7
correct_answer        0
correctness           0
tags                  0
elapsed_time          0
elapsed_seconds       0
attempt_index         0
time_delta_seconds    0
log_elapsed_time      0
log_time_delta        0
dtype: int64

Correctness distribution:
correctness
1    2196
0    1172
Name: count, dtype: int64

Unique students:
10

Unique questions:
2320

Unique tags:
183

First 10 processed rows:


,user_id,timestamp,solving_id,question_id,user_answer,correct_answer,correctness,tags,elapsed_time,elapsed_seconds,attempt_index,time_delta_seconds,log_elapsed_time,log_time_delta
0,u1,1565096190868,1,q5012,b,c,0,74,38000,38.0,1,0.000,3.663562,0.000000
1,u1,1565096221062,2,q4706,c,c,1,71,24000,24.0,1,30.194,3.218876,3.440226
2,u1,1565096293432,3,q4366,b,b,1,103,68000,68.0,1,72.370,4.234107,4.295515
3,u1,1565096339668,4,q4829,a,c,0,83,42000,42.0,1,46.236,3.761200,3.855156
4,u1,1565096401774,5,q6528,b,d,0,90,59000,59.0,1,62.106,4.094345,4.144816
5,u1,1565096463370,6,q4793,a,b,0,82,58000,58.0,1,61.596,4.077537,4.136701
6,u1,1565096501746,7,q6488,a,a,1,77,35000,35.0,1,38.376,3.583519,3.673156
7,u1,1565097101361,8,q356,b,b,1,30;34;42;181;183,23000,23.0,1,599.615,3.178054,6.397954
8,u1,1565097171393,9,q1382,c,c,1,24;43;184;181,22000,22.0,1,70.032,3.135494,4.263130
9,u1,1565097240758,10,q830,b,b,1,27;28;45;184;182,25000,25.0,1,69.365,3.258097,4.253696



Saved to:
D:\Lokesh files\Projects\LearningDNA\processed\kt1_processed_sample.csv
